# WOP vs WoS (C++ only, unit cube)

Notebook compares three C++ configurations via existing `wop_cli`:
- `wop_escape`
- `wop_project`
- `wos`

Two views are reported:
- fixed-N comparison
- time-to-eps comparison


In [ ]:
from __future__ import annotations

import json
import os
import statistics
import subprocess
import time
from pathlib import Path

import pandas as pd


def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root with external_wop_cpp")


WORKSPACE_ROOT = find_workspace_root(Path.cwd())
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"


def find_cpp_cli() -> Path:
    env_cli = os.environ.get("WOP_CPP_CLI")
    if env_cli:
        p = Path(env_cli).expanduser().resolve()
        if p.exists():
            return p

    candidates = [
        CPP_ROOT / "build" / "wop_cli",
        CPP_ROOT / "build" / "wop_cli.exe",
        CPP_ROOT / "build" / "Release" / "wop_cli.exe",
        CPP_ROOT / "build" / "Debug" / "wop_cli.exe",
    ]
    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(
        "wop_cli not found. Build external_wop_cpp first or set WOP_CPP_CLI env var."
    )


CPP_CLI = find_cpp_cli()
print(f"Using CLI: {CPP_CLI}")


In [ ]:
X0 = (3.0, 0.0, 0.0)
N_VALUES = [100, 1000, 10000, 100000, 1000000]
SEEDS = [314159, 271828, 161803]
MAX_STEPS = 200000
WARMUP_RUNS = 1
REPEATS = 3
EPS_TARGETS = [2e-1, 1e-1, 5e-2, 2e-2]

CONFIGS = {
    "wop_escape": {
        "method": "wop",
        "args": ["--r-max", "1000000", "--r-max-mode", "escape", "--r-max-factor", "3.0"],
    },
    "wop_project": {
        "method": "wop",
        "args": ["--r-max", "0", "--r-max-mode", "project", "--r-max-factor", "3.0"],
    },
    "wos": {
        "method": "wos",
        "args": ["--delta", "1e-3", "--rho-scale", "1.0", "--rho1-scale", "2.0"],
    },
}


In [ ]:
def run_cpp_once(config_name: str, n_paths: int, seed: int) -> dict:
    cfg = CONFIGS[config_name]
    cmd = [
        str(CPP_CLI),
        "--example", "box",
        "--method", str(cfg["method"]),
        "--x0", f"{X0[0]} {X0[1]} {X0[2]}",
        "--n", str(int(n_paths)),
        "--seed", str(int(seed)),
        "--max-steps", str(int(MAX_STEPS)),
        "--json",
        *[str(x) for x in cfg["args"]],
    ]

    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0

    payload = json.loads(out)
    payload["config"] = config_name
    payload["wall_time_sec"] = float(elapsed)
    return payload


def run_with_repeats(config_name: str, n_paths: int, seed: int) -> dict:
    for _ in range(WARMUP_RUNS):
        _ = run_cpp_once(config_name=config_name, n_paths=n_paths, seed=seed)

    runs = [run_cpp_once(config_name=config_name, n_paths=n_paths, seed=seed) for _ in range(REPEATS)]
    time_values = [float(r["wall_time_sec"]) for r in runs]

    ref = dict(runs[0])
    ref["wall_time_sec"] = float(statistics.median(time_values))
    return ref


In [ ]:
rows = []
for config_name in CONFIGS:
    for n_paths in N_VALUES:
        for seed in SEEDS:
            row = run_with_repeats(config_name=config_name, n_paths=n_paths, seed=seed)
            row["seed"] = int(seed)
            rows.append(row)
            print(
                f"config={config_name:>11} n={n_paths:>7} seed={seed} "
                f"J={float(row['J']):.8f} abs_error={float(row['abs_error']):.3e} "
                f"eps={float(row['eps']):.3e} time={float(row['wall_time_sec']):.3f}s"
            )

df_raw = pd.DataFrame(rows)
df_raw


In [ ]:
fixed_n = (
    df_raw.groupby(["config", "n_total"], as_index=False)
    .agg(
        abs_error_median=("abs_error", "median"),
        eps_median=("eps", "median"),
        time_median_sec=("wall_time_sec", "median"),
        mean_steps_median=("mean_steps", "median"),
        trunc_rate=("n_truncated", lambda s: float(s.sum()) / float((df_raw.loc[s.index, "n_total"]).sum())),
    )
    .sort_values(["n_total", "config"])
    .reset_index(drop=True)
)

print("Fixed-N comparison")
fixed_n


In [ ]:
records = []
for config_name, grp in fixed_n.groupby("config"):
    grp_sorted = grp.sort_values("n_total")
    for eps_target in EPS_TARGETS:
        ok = grp_sorted[grp_sorted["eps_median"] <= eps_target]
        if len(ok) == 0:
            records.append({
                "config": config_name,
                "eps_target": float(eps_target),
                "min_time_sec": float("nan"),
                "n_at_target": int(-1),
            })
            continue

        best = ok.iloc[ok["time_median_sec"].argmin()]
        records.append({
            "config": config_name,
            "eps_target": float(eps_target),
            "min_time_sec": float(best["time_median_sec"]),
            "n_at_target": int(best["n_total"]),
        })

time_to_eps = pd.DataFrame(records).sort_values(["eps_target", "config"]).reset_index(drop=True)
print("Time-to-eps comparison")
time_to_eps


In [ ]:
artifacts_dir = WORKSPACE_ROOT / "tmp" / "jupyter-notebook" / "wop-vs-wos-cpp-unit-cube"
artifacts_dir.mkdir(parents=True, exist_ok=True)
(artifacts_dir / "raw_rows.json").write_text(df_raw.to_json(orient="records", indent=2), encoding="utf-8")
(artifacts_dir / "fixed_n.json").write_text(fixed_n.to_json(orient="records", indent=2), encoding="utf-8")
(artifacts_dir / "time_to_eps.json").write_text(time_to_eps.to_json(orient="records", indent=2), encoding="utf-8")
print("Saved artifacts to", artifacts_dir)
